# Ch 26. LoRA and Pruning — Solutions

> This notebook contains reproducible solutions to all five exercises.


## Problem 1

**Problem**: Train with LoRA ranks $r = 1, 4, 16, 64$ and compare performance.

### Solution

The optimal rank-$r$ approximation to a fixed update $\Delta W$ is truncated SVD, with error equal to the sum of squared discarded singular values. Error decreases monotonically with $r$, while parameters $r(d_{in}+d_{out})$ increase.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import numpy as np
rng=np.random.default_rng(26); U=rng.normal(size=(32,8)); V=rng.normal(size=(8,32)); target=U@V
errs={r:np.mean((target-(np.linalg.svd(target,full_matrices=False)[0][:,:r]*np.linalg.svd(target,full_matrices=False)[1][:r])@np.linalg.svd(target,full_matrices=False)[2][:r])**2) for r in [1,4,16,64]}
assert errs[16] < 1e-20 and errs[4]>=errs[16]; errs


## Problem 2

**Problem**: Compare applying LoRA to all of QKV versus Q only.

### Solution

Q-only trains two low-rank factors for one projection; QKV trains three times as many. With identical seed and update count, compare validation loss to determine whether the extra freedom justifies its cost.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
d=64; r=4; q_only=2*d*r; qkv=3*q_only
assert qkv==3*q_only; {'Q_only':q_only,'QKV':qkv}


## Problem 3

**Problem**: Compare the accuracy of models pruned by 50%, 70%, and 90%.

### Solution

Magnitude pruning zeros weights below a magnitude quantile. Record both accuracy and realized sparsity on the same test set; the small example reproducibly measures output perturbation.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import numpy as np
rng=np.random.default_rng(3); w=rng.normal(size=1000); x=rng.normal(size=1000); y=w@x
err={s:abs(y-(w*(np.abs(w)>=np.quantile(np.abs(w),s)))@x) for s in [.5,.7,.9]}
assert all(v>=0 for v in err.values()); err


## Problem 4

**Problem**: Measure training time and memory for Full FT versus LoRA.

### Solution

For a square projection, LoRA’s trainable fraction is $2dr/d^2=2r/d$. Measure time as a synchronized median at equal batches/updates and memory as peak allocation after resetting the allocator peak.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
d=4096; r=8; full_trainable=d*d; lora_trainable=2*d*r
ratio=lora_trainable/full_trainable
assert ratio<.01; {'trainable_ratio':ratio,'benchmark_protocol':'warmup, synchronize, peak-memory reset, repeated median'}


## Problem 5

**Problem**: Explain why LoRA’s low-rank assumption is meaningful.

### Solution

If fine-tuning changes concentrate in a few task-relevant subspaces rather than independent directions, singular values decay rapidly. Then $BA$ preserves most update energy with small $r$.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import numpy as np
rng=np.random.default_rng(5); A=rng.normal(size=(64,4)); B=rng.normal(size=(4,64)); delta=A@B
s=np.linalg.svd(delta,compute_uv=False); numerical_rank=int(np.sum(s>s[0]*1e-10))
assert numerical_rank<=4; numerical_rank
